# Phase 4 - Notebook 01: Cost Volume & Plane Sweeping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/01_cost_volume_plane_sweeping.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand Multi-View Stereo (MVS) fundamentals and stereo matching
2. Implement the Plane Sweeping algorithm step by step
3. Build a Cost Volume and visualize it
4. Implement differentiable depth regression via soft argmin
5. Understand how Cost Volume connects to MVSplat's geometry reasoning

**Estimated Time**: 75 minutes

**Prerequisites**: Phase 1 (camera model basics), linear algebra

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Multi-View Stereo (MVS) Fundamentals

### 1.1 The Stereo Matching Problem

Given two images of the same scene from different viewpoints, **stereo matching** finds the corresponding pixel in the second image for each pixel in the first image.

```
Left Image           Right Image
┌─────────┐          ┌─────────┐
│    *     │  ──────► │      *  │   Same 3D point
│  (u,v)  │          │ (u',v') │   appears at different
└─────────┘          └─────────┘   pixel locations

Disparity d = u - u'  →  Depth Z = f·B / d
  (larger disparity = closer object)
```

### 1.2 From Stereo to Multi-View

| Aspect | Stereo (2 views) | Multi-View (N views) |
|--------|------------------|---------------------|
| Input | Rectified pair | Arbitrary views |
| Search | 1D (along epipolar line) | 2D (homography warp) |
| Cost | SAD/SSD/NCC | Learned features |
| Output | Disparity map | Depth map |

MVSplat uses the **multi-view** formulation with **Plane Sweeping**.

In [ ]:
# Visualize the stereo matching concept

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

np.random.seed(42)

# Create a simple synthetic scene (top view)
ax = axes[0]
ax.set_title('Scene (Top View)', fontsize=12, fontweight='bold')

# 3D points
scene_x = np.array([-1, 0, 1.5, -0.5, 0.8])
scene_z = np.array([3, 5, 4, 6, 3.5])
ax.scatter(scene_x, scene_z, s=100, c='blue', zorder=5, label='3D Points')

# Two cameras
cam1_x, cam1_z = -1.5, 0
cam2_x, cam2_z = 1.5, 0
ax.scatter([cam1_x, cam2_x], [cam1_z, cam2_z], s=200, c='red', marker='^', zorder=5)
ax.text(cam1_x, cam1_z - 0.5, 'Cam 1', ha='center', fontsize=10)
ax.text(cam2_x, cam2_z - 0.5, 'Cam 2', ha='center', fontsize=10)

# Projection rays
for px, pz in zip(scene_x, scene_z):
    ax.plot([cam1_x, px], [cam1_z, pz], 'r--', alpha=0.3, lw=1)
    ax.plot([cam2_x, px], [cam2_z, pz], 'g--', alpha=0.3, lw=1)

ax.set_xlabel('X')
ax.set_ylabel('Z (depth)')
ax.set_xlim(-3, 3)
ax.set_ylim(-1, 8)
ax.legend()
ax.grid(True, alpha=0.3)

# Simulated image views
for idx, (ax, title, offset) in enumerate([(axes[1], 'Camera 1 View', 0), 
                                            (axes[2], 'Camera 2 View', 0.3)]):
    ax.set_title(title, fontsize=12, fontweight='bold')
    img = np.ones((10, 10, 3)) * 0.9
    # Simulate projected points at different positions
    positions = [3+offset, 5, 7-offset, 2+offset, 6.5-offset]
    for i, pos in enumerate(positions):
        ax.plot(pos, 5, 'o', markersize=15, color='blue', alpha=0.7)
        if idx == 0:
            ax.annotate(f'P{i+1}', (pos, 5), textcoords='offset points',
                       xytext=(0, 12), ha='center', fontsize=8)
    
    # Draw epipolar line
    ax.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Epipolar line')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Same 3D points project to DIFFERENT pixel locations in each view.")
print("The displacement (disparity) encodes depth information.")

## 2. Plane Sweeping Algorithm

### 2.1 Core Idea

Instead of searching for correspondences directly, **Plane Sweeping** tests a set of **depth hypotheses**:

```
For each depth hypothesis d:
  1. Assume ALL pixels in the reference image are at depth d
  2. Project them to 3D, then reproject to the source image
  3. Compare features: if they match → correct depth!

                  depth d1    d2    d3    d4
Camera ──────────┤─────┤─────┤─────┤─────┤──── Scene
                  near                    far
```

### 2.2 Depth Hypotheses Sampling

Two common strategies:
- **Uniform**: Equal spacing between d_min and d_max
- **Log-uniform**: Denser near camera (where depth precision matters more)

In [ ]:
# Compare uniform vs log-uniform depth sampling

import math

num_depths = 32
d_min, d_max = 0.5, 50.0

# Uniform sampling
depths_uniform = np.linspace(d_min, d_max, num_depths)

# Log-uniform sampling
depths_log = np.exp(np.linspace(math.log(d_min), math.log(d_max), num_depths))

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Plot uniform
ax = axes[0]
ax.scatter(depths_uniform, np.ones_like(depths_uniform), s=50, c='blue', alpha=0.7)
for d in depths_uniform:
    ax.axvline(x=d, color='blue', alpha=0.15, lw=1)
ax.set_title('Uniform Depth Sampling', fontsize=11, fontweight='bold')
ax.set_xlabel('Depth')
ax.set_yticks([])
ax.set_xlim(0, d_max + 2)

# Plot log-uniform
ax = axes[1]
ax.scatter(depths_log, np.ones_like(depths_log), s=50, c='green', alpha=0.7)
for d in depths_log:
    ax.axvline(x=d, color='green', alpha=0.15, lw=1)
ax.set_title('Log-uniform Depth Sampling (used in MVSplat)', fontsize=11, fontweight='bold')
ax.set_xlabel('Depth')
ax.set_yticks([])
ax.set_xlim(0, d_max + 2)

plt.tight_layout()
plt.show()

print(f"Uniform: spacing = {np.diff(depths_uniform).mean():.2f} (constant)")
print(f"Log-uniform: near spacing = {depths_log[1]-depths_log[0]:.3f}, "
      f"far spacing = {depths_log[-1]-depths_log[-2]:.2f}")
print("\nLog-uniform puts MORE depth hypotheses near the camera,")
print("where small depth changes cause large pixel shifts (higher precision needed).")

## 3. Homography Warping

### 3.1 Key Formula

For a fronto-parallel plane at depth $d$, the **homography** mapping pixels from the reference to the source view is:

$$H = K_{src} \cdot \left( R + \frac{t \cdot n^T}{d} \right) \cdot K_{ref}^{-1}$$

where:
- $K_{ref}, K_{src}$: camera intrinsics
- $R, t$: relative rotation and translation
- $n = [0, 0, 1]^T$: plane normal (fronto-parallel)
- $d$: depth hypothesis

### 3.2 Step-by-step Process

```
For pixel (u, v) in reference image at depth d:

  1. Back-project to 3D:  P = d · K_ref^{-1} · [u, v, 1]^T
  2. Transform to source:  P' = R · P + t
  3. Project to source:    [u', v'] = K_src · P' / P'_z
  4. Sample feature:       F_src(u', v')  (bilinear interpolation)
```

In [ ]:
def create_camera_intrinsics(fx, fy, cx, cy):
    """Create 3x3 camera intrinsics matrix."""
    K = torch.tensor([
        [fx, 0, cx],
        [0, fy, cy],
        [0,  0,  1]
    ], dtype=torch.float32)
    return K


def create_relative_pose(tx=0.2, ty=0.0, tz=0.0, angle_deg=5.0):
    """Create a 4x4 relative pose (small baseline)."""
    angle = np.radians(angle_deg)
    R = torch.tensor([
        [np.cos(angle), 0, np.sin(angle)],
        [0, 1, 0],
        [-np.sin(angle), 0, np.cos(angle)]
    ], dtype=torch.float32)
    t = torch.tensor([tx, ty, tz], dtype=torch.float32)
    T = torch.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T


def homography_warp_educational(pixel_u, pixel_v, depth, K_ref, K_src, T_src_ref):
    """
    Warp a single pixel from reference to source view at given depth.
    Educational version - step by step.
    """
    # Step 1: Back-project to 3D
    pixel_homo = torch.tensor([pixel_u, pixel_v, 1.0])
    K_ref_inv = torch.inverse(K_ref)
    ray = K_ref_inv @ pixel_homo
    point_3d = depth * ray
    
    # Step 2: Transform to source camera frame
    R = T_src_ref[:3, :3]
    t = T_src_ref[:3, 3]
    point_src = R @ point_3d + t
    
    # Step 3: Project to source image
    projected = K_src @ point_src
    u_src = projected[0] / projected[2]
    v_src = projected[1] / projected[2]
    
    return u_src.item(), v_src.item(), point_3d.numpy()


# Setup cameras
H, W = 16, 16
fx, fy = 20.0, 20.0
cx, cy = W / 2.0, H / 2.0

K_ref = create_camera_intrinsics(fx, fy, cx, cy)
K_src = create_camera_intrinsics(fx, fy, cx, cy)  # Same intrinsics
T_src_ref = create_relative_pose(tx=0.5, angle_deg=3.0)

# Demonstrate warping for one pixel at different depths
pixel_u, pixel_v = 10.0, 8.0
test_depths = [1.0, 2.0, 3.0, 5.0, 10.0]

print(f"Reference pixel: ({pixel_u}, {pixel_v})")
print(f"Camera baseline: tx=0.5, rotation=3 degrees")
print(f"\n{'Depth':>6s} | {'Source u':>10s} | {'Source v':>10s} | {'3D Point':>25s}")
print("-" * 60)

warped_points = []
for d in test_depths:
    u_src, v_src, p3d = homography_warp_educational(pixel_u, pixel_v, d, K_ref, K_src, T_src_ref)
    warped_points.append((d, u_src, v_src, p3d))
    print(f"{d:6.1f} | {u_src:10.3f} | {v_src:10.3f} | [{p3d[0]:.2f}, {p3d[1]:.2f}, {p3d[2]:.2f}]")

print("\nKey insight: The same pixel maps to DIFFERENT source locations at each depth.")
print("Only the CORRECT depth produces a matching feature!")

In [ ]:
# Visualize the warping at different depths

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: 3D view of the warping
ax = axes[0]
ax.set_title('Plane Sweeping: 3D View (Top-down)', fontsize=12, fontweight='bold')

# Camera positions
ax.scatter([0], [0], s=200, c='red', marker='^', zorder=5, label='Ref Camera')
ax.scatter([0.5], [0], s=200, c='green', marker='^', zorder=5, label='Src Camera')

# Depth planes
colors = plt.cm.viridis(np.linspace(0, 1, len(test_depths)))
for i, (d, u_s, v_s, p3d) in enumerate(warped_points):
    # Draw depth plane
    ax.axhline(y=d, color=colors[i], alpha=0.3, lw=2, linestyle='--')
    ax.text(2.5, d + 0.15, f'd={d}', fontsize=8, color=colors[i])
    
    # Draw ray from ref camera to 3D point
    ax.plot([0, p3d[0]], [0, p3d[2]], '-', color='red', alpha=0.3, lw=1)
    
    # 3D point
    ax.scatter([p3d[0]], [p3d[2]], s=60, c=[colors[i]], zorder=5)

ax.set_xlabel('X')
ax.set_ylabel('Z (depth)')
ax.set_xlim(-1, 3)
ax.set_ylim(-0.5, 12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Right: Source image showing warped locations
ax = axes[1]
ax.set_title('Source Image: Warped Pixel Locations', fontsize=12, fontweight='bold')

# Draw image boundary
ax.plot([0, W, W, 0, 0], [0, 0, H, H, 0], 'k-', lw=2)

# Reference pixel position
ax.scatter([pixel_u], [pixel_v], s=200, c='red', marker='*', zorder=5, label='Ref pixel')

# Warped positions at each depth
for i, (d, u_s, v_s, _) in enumerate(warped_points):
    ax.scatter([u_s], [v_s], s=80, c=[colors[i]], zorder=5)
    ax.annotate(f'd={d}', (u_s, v_s), textcoords='offset points',
               xytext=(10, 5), fontsize=8, color=colors[i])

# Connect with line
u_vals = [wp[1] for wp in warped_points]
v_vals = [wp[2] for wp in warped_points]
ax.plot(u_vals, v_vals, '--', color='gray', alpha=0.5, label='Sweep path')

ax.set_xlabel('u (pixels)')
ax.set_ylabel('v (pixels)')
ax.set_xlim(-2, W + 2)
ax.set_ylim(H + 2, -2)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("At each depth hypothesis, the reference pixel maps to a different source location.")
print("The correct depth gives the best feature match.")

## 4. Building the Cost Volume

### 4.1 Cost Volume Definition

The **Cost Volume** $C \in \mathbb{R}^{H \times W \times D}$ stores the matching cost for each pixel $(u, v)$ at each depth hypothesis $d_k$:

$$C(u, v, d_k) = \| F_{ref}(u, v) - F_{src}(u'_k, v'_k) \|^2$$

where $(u'_k, v'_k)$ is the warped pixel location at depth $d_k$.

**Interpretation:**
- Low cost → features match → likely correct depth
- High cost → features don't match → wrong depth

### 4.2 Implementation

In [ ]:
def build_cost_volume_simple(feat_ref, feat_src, K_ref, K_src, T_src_ref, 
                             depth_planes):
    """
    Build a cost volume via plane sweeping.
    
    Educational implementation - clear but not optimized.
    
    Args:
        feat_ref: [C, H, W] reference feature map
        feat_src: [C, H, W] source feature map
        K_ref, K_src: [3, 3] camera intrinsics
        T_src_ref: [4, 4] relative pose
        depth_planes: [D] depth hypotheses
    
    Returns:
        cost_volume: [D, H, W] matching cost
    """
    C, H, W = feat_ref.shape
    D = len(depth_planes)
    
    # Extract pose components
    R = T_src_ref[:3, :3]
    t = T_src_ref[:3, 3:]
    K_ref_inv = torch.inverse(K_ref)
    
    # Create pixel grid
    u = torch.arange(W, dtype=torch.float32)
    v = torch.arange(H, dtype=torch.float32)
    vv, uu = torch.meshgrid(v, u, indexing='ij')
    ones = torch.ones_like(uu)
    pixels = torch.stack([uu, vv, ones], dim=0)  # [3, H, W]
    pixels_flat = pixels.reshape(3, -1)  # [3, H*W]
    
    # Compute rays in reference camera
    rays = K_ref_inv @ pixels_flat  # [3, H*W]
    
    cost_volume = torch.zeros(D, H, W)
    
    for i, depth in enumerate(depth_planes):
        # Back-project to 3D at this depth
        points_3d = depth * rays  # [3, H*W]
        
        # Transform to source camera
        points_src = R @ points_3d + t  # [3, H*W]
        
        # Project to source image
        proj = K_src @ points_src  # [3, H*W]
        u_src = proj[0] / (proj[2] + 1e-8)  # [H*W]
        v_src = proj[1] / (proj[2] + 1e-8)
        
        # Normalize to [-1, 1] for grid_sample
        u_norm = 2.0 * u_src / (W - 1) - 1.0
        v_norm = 2.0 * v_src / (H - 1) - 1.0
        grid = torch.stack([u_norm, v_norm], dim=-1)  # [H*W, 2]
        grid = grid.reshape(1, H, W, 2)  # [1, H, W, 2]
        
        # Sample source features
        feat_src_4d = feat_src.unsqueeze(0)  # [1, C, H, W]
        warped = F.grid_sample(feat_src_4d, grid, mode='bilinear', 
                              padding_mode='zeros', align_corners=True)
        warped = warped.squeeze(0)  # [C, H, W]
        
        # Matching cost: squared difference
        cost = (feat_ref - warped).pow(2).mean(dim=0)  # [H, W]
        cost_volume[i] = cost
    
    return cost_volume


# Create synthetic feature maps with a known depth pattern
H, W = 32, 32
C = 8  # Feature channels

# Reference features: random but spatially coherent
torch.manual_seed(42)
feat_ref = F.interpolate(
    torch.randn(1, C, 8, 8), size=(H, W), mode='bilinear', align_corners=True
).squeeze(0)

# Ground truth depth: simple plane + bump
u_grid = torch.linspace(-1, 1, W)
v_grid = torch.linspace(-1, 1, H)
VV, UU = torch.meshgrid(v_grid, u_grid, indexing='ij')
gt_depth = 5.0 + 1.0 * torch.sin(UU * 3) + 0.5 * torch.cos(VV * 2)

# Source features: warp ref features using ground truth depth
# (In practice, these come from a real second image)
K = create_camera_intrinsics(50.0, 50.0, W/2.0, H/2.0)
T = create_relative_pose(tx=0.3, angle_deg=2.0)

# For this demo, create source features by warping reference
feat_src = feat_ref.clone()  # Simplified: same features

# Build depth planes
num_depths = 48
depth_planes = torch.exp(torch.linspace(
    math.log(2.0), math.log(10.0), num_depths
))

# Build cost volume
print("Building cost volume...")
cost_volume = build_cost_volume_simple(feat_ref, feat_src, K, K, T, depth_planes)
print(f"Cost volume shape: {cost_volume.shape}  (D={num_depths}, H={H}, W={W})")
print(f"Cost range: [{cost_volume.min():.4f}, {cost_volume.max():.4f}]")

In [ ]:
# Visualize the cost volume

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Show cost volume slices at different depths
slice_indices = [0, num_depths//6, num_depths//3, num_depths//2, 
                 2*num_depths//3, 5*num_depths//6, num_depths-2, num_depths-1]

for idx, (ax, si) in enumerate(zip(axes.flat, slice_indices)):
    im = ax.imshow(cost_volume[si].numpy(), cmap='RdYlGn_r', 
                   vmin=0, vmax=cost_volume.max().item() * 0.5)
    ax.set_title(f'd = {depth_planes[si]:.2f}', fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Cost Volume Slices at Different Depth Hypotheses\n'
             '(Green = low cost = good match, Red = high cost = poor match)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each slice shows matching cost at a specific depth hypothesis.")
print("Low cost (green) regions indicate pixels whose true depth is near that hypothesis.")

In [ ]:
# Visualize cost profile for specific pixels

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

test_pixels = [(8, 8), (16, 16), (24, 12)]
colors = ['#D32F2F', '#1565C0', '#2E7D32']

for ax_idx, ((py, px), color) in enumerate(zip(test_pixels, colors)):
    ax = axes[ax_idx]
    
    # Extract cost profile for this pixel
    cost_profile = cost_volume[:, py, px].numpy()
    depths_np = depth_planes.numpy()
    
    # Plot cost vs depth
    ax.plot(depths_np, cost_profile, '-o', color=color, markersize=3, lw=1.5)
    ax.fill_between(depths_np, cost_profile, alpha=0.1, color=color)
    
    # Mark minimum
    min_idx = np.argmin(cost_profile)
    ax.axvline(x=depths_np[min_idx], color='orange', linestyle='--', lw=2,
              label=f'Best depth = {depths_np[min_idx]:.2f}')
    ax.scatter([depths_np[min_idx]], [cost_profile[min_idx]], s=100, 
             color='orange', zorder=5)
    
    # Mark GT depth
    gt_d = gt_depth[py, px].item()
    ax.axvline(x=gt_d, color='green', linestyle=':', lw=2,
              label=f'GT depth = {gt_d:.2f}')
    
    ax.set_xlabel('Depth', fontsize=10)
    ax.set_ylabel('Matching Cost', fontsize=10)
    ax.set_title(f'Pixel ({px}, {py})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cost Profile per Pixel (Cost vs Depth Hypothesis)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The cost profile shows a 'valley' at the correct depth.")
print("The soft argmin extracts depth by computing the weighted average.")

## 5. Depth Regression via Soft Argmin

### 5.1 From Cost to Depth

Instead of taking the hard argmin (non-differentiable), we use **soft argmin**:

$$p(d_k | u, v) = \frac{\exp(-C(u,v,d_k) / \tau)}{\sum_{j} \exp(-C(u,v,d_j) / \tau)}$$

$$\hat{d}(u, v) = \sum_{k=1}^{D} d_k \cdot p(d_k | u, v)$$

where $\tau$ is a temperature parameter:
- High $\tau$: soft, smooth distribution (more regularized)
- Low $\tau$: sharp, peaked distribution (closer to argmin)

In [ ]:
def soft_argmin_depth(cost_volume, depth_planes, temperature=1.0):
    """
    Differentiable depth regression from cost volume.
    
    Args:
        cost_volume: [D, H, W]
        depth_planes: [D]
        temperature: softmax temperature
    
    Returns:
        depth_map: [H, W]
        prob_volume: [D, H, W] depth probability distribution
    """
    D, H, W = cost_volume.shape
    
    # Convert cost to probability (low cost = high probability)
    neg_cost = -cost_volume / temperature  # [D, H, W]
    prob_volume = F.softmax(neg_cost, dim=0)  # [D, H, W]
    
    # Expected depth (weighted sum)
    depth_weights = depth_planes.reshape(D, 1, 1)  # [D, 1, 1]
    depth_map = (prob_volume * depth_weights).sum(dim=0)  # [H, W]
    
    return depth_map, prob_volume


# Compare different temperatures
temperatures = [0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(2, len(temperatures), figsize=(18, 8))

for col, temp in enumerate(temperatures):
    depth_map, prob_vol = soft_argmin_depth(cost_volume, depth_planes, temp)
    
    # Top row: depth map
    ax = axes[0, col]
    im = ax.imshow(depth_map.numpy(), cmap='plasma')
    ax.set_title(f'Depth (T={temp})', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')
    
    # Bottom row: probability distribution for center pixel
    ax = axes[1, col]
    prob_profile = prob_vol[:, H//2, W//2].numpy()
    ax.bar(depth_planes.numpy(), prob_profile, width=0.1, alpha=0.7, color='steelblue')
    ax.set_xlabel('Depth')
    ax.set_ylabel('Probability')
    ax.set_title(f'P(d) at center (T={temp})', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Temperature on Soft Argmin Depth Regression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Temperature controls the sharpness of the depth distribution:")
print("  T=0.01: Very sharp (almost hard argmin, less smooth gradients)")
print("  T=0.1:  Sharp but differentiable (common in practice)")
print("  T=1.0:  Smooth (good for training stability)")
print("  T=10.0: Very smooth (over-regularized)")

## 6. Using Our `src/feedforward` Module

Let's now use the `CostVolumeBuilder` from our project's source code.

In [ ]:
from src.feedforward.cost_volume import (
    create_depth_planes,
    CostVolumeBuilder,
    depth_regression_softargmin,
)

# Create the builder
builder = CostVolumeBuilder(
    num_depths=32,
    min_depth=2.0,
    max_depth=10.0,
    sampling='log_uniform'
)

print(f"CostVolumeBuilder created:")
print(f"  Depth planes: {builder.depth_planes.shape[0]} planes")
print(f"  Range: [{builder.depth_planes[0]:.2f}, {builder.depth_planes[-1]:.2f}]")
print(f"  First 5: {builder.depth_planes[:5].tolist()}")

# Create batched inputs
B, C, H, W = 1, 16, 16, 16
feat_ref = torch.randn(B, C, H, W)
feat_src = torch.randn(B, C, H, W)
K_ref = create_camera_intrinsics(30, 30, W/2, H/2).unsqueeze(0)  # [1, 3, 3]
K_src = K_ref.clone()
T = create_relative_pose(tx=0.3).unsqueeze(0)  # [1, 4, 4]

# Build cost volume (single source view)
cv = builder(feat_ref, [feat_src], K_ref, [K_src], [T])
print(f"\nCost volume shape: {cv.shape}  (B, C, D, H, W)")

# Regress depth
depth, prob = depth_regression_softargmin(cv, builder.depth_planes, temperature=0.5)
print(f"Predicted depth shape: {depth.shape}")
print(f"Depth range: [{depth.min():.2f}, {depth.max():.2f}]")

## 7. Cost Volume in MVSplat vs Classical MVS

### Key Differences

| Aspect | Classical MVS (MVSNet) | MVSplat |
|--------|----------------------|--------|
| Feature extractor | Fixed CNN | Learned U-Net |
| Cost Volume processing | 3D CNN regularization | 3D U-Net |
| Output | Depth map only | Depth + Gaussian params |
| Training signal | Depth supervision | Photometric (rendering) loss |
| Application | Dense reconstruction | Novel view synthesis |

### In MVSplat

The Cost Volume is not just used for depth estimation - it provides **geometric features** that are used to predict ALL Gaussian parameters:

```
Cost Volume  ──►  3D U-Net  ──►  Processed Features
                                       │
                        ┌───────────────┼───────────────┐
                        ▼               ▼               ▼
                   Depth Head      Covariance Head  Opacity Head
                        │               │               │
                        ▼               ▼               ▼
                  Depth map        Scale + Rot       Opacity
                        │               │               │
                        └───────────────┴───────────────┘
                                        │
                                        ▼
                              Pixel-aligned Gaussians
```

In [ ]:
# Summary

summary = """
╔═══════════════════════════════════════════════════════════════════════╗
║              Notebook 01 Summary: Cost Volume & Plane Sweeping       ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. STEREO MATCHING                                                  ║
║     - Find corresponding pixels across views                        ║
║     - Disparity → depth relationship                                ║
║                                                                      ║
║  2. PLANE SWEEPING                                                   ║
║     - Sample D depth hypotheses between d_min and d_max             ║
║     - Log-uniform: denser near camera (higher precision)            ║
║     - At each depth: warp source features via homography            ║
║                                                                      ║
║  3. COST VOLUME                                                      ║
║     - C(u,v,d) = ||F_ref(u,v) - F_src(u',v')||^2                   ║
║     - Shape: [D, H, W] (or [B, C, D, H, W] with features)          ║
║     - Low cost = good match = correct depth                         ║
║                                                                      ║
║  4. SOFT ARGMIN                                                      ║
║     - Differentiable depth regression                               ║
║     - p(d) = softmax(-cost / temperature)                           ║
║     - depth = sum(d_k * p(d_k))                                     ║
║     - Temperature controls sharpness                                ║
║                                                                      ║
║  5. IN MVSplat                                                       ║
║     - Cost Volume → 3D U-Net → Gaussian parameter prediction        ║
║     - Not just depth, but also scale, rotation, opacity             ║
║                                                                      ║
╚═══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

## What's Next?

In the next notebook, we'll explore **Pixel-aligned Gaussian Representation** - how the predicted depth is used to create structured 3D Gaussians:

**[02_pixel_aligned_gaussians.ipynb](./02_pixel_aligned_gaussians.ipynb)**

---

## References

1. MVSNet: https://arxiv.org/abs/1804.02505
2. MVSplat: https://arxiv.org/abs/2403.14627
3. Plane Sweeping Stereo: Collins (1996)